In [2]:
from transformers import (
    AutoProcessor,
    Qwen2AudioForConditionalGeneration,
)
from peft import PeftModel
import torch
import librosa

# from dotenv import load_dotenv


import os
import gc
import json
from IPython.display import Audio
from json_repair import repair_json

In [ ]:
load_dotenv()

True

In [5]:
import numpy as np

In [6]:
wav = np.load("/home/studio/work/audio_stream/debug_samples/20260507_123335_334042/audio.npy")
wav = np.load("/home/studio/work/audio_stream/debug_samples/20260507_123338_298905/audio.npy")

In [7]:
def load_model() -> tuple:
    """Load Qwen2Audio processor and LoRA-adapted model.

    Reads `MODEL_NAME` and `LORA_PATH` from a `.env` file or environment.

    Returns:
        (processor, model)
    """
    # Load env vars if present
    try:
        load_dotenv()
    except Exception:
        # noop if dotenv not available
        pass

    model_name = os.getenv("MODEL_NAME") or os.getenv("QWEN2AUDIO_MODEL_NAME")
    lora_path = os.getenv("LORA_PATH") or os.getenv("QWEN2AUDIO_LORA_PATH")
    model_cache_path = os.getenv("QWEN_CACHE_DIR")

    if not model_name:
        raise RuntimeError("MODEL_NAME not set in environment or .env")
    if not lora_path:
        raise RuntimeError("LORA_PATH not set in environment or .env")

    LOGGER.info("Loading model and processor... model=%s lora=%s", model_name, lora_path)

    processor = AutoProcessor.from_pretrained(model_name, cache_dir=model_cache_path)

    model = Qwen2AudioForConditionalGeneration.from_pretrained(
        model_name, torch_dtype="auto", device_map="auto", cache_dir=model_cache_path
    )

    model = PeftModel.from_pretrained(model, lora_path)
    model.eval()

    LOGGER.info("Model and processor loaded successfully")
    return processor, model


In [12]:
def load_model() -> tuple:
    """Load Qwen2Audio processor and LoRA-adapted model.

    Reads `MODEL_NAME` and `LORA_PATH` from a `.env` file or environment.

    Returns:
        (processor, model)
    """
    # Load env vars if present
    try:
        load_dotenv()
    except Exception:
        # noop if dotenv not available
        pass

    model_name = os.getenv("MODEL_NAME") or os.getenv("QWEN2AUDIO_MODEL_NAME")
    lora_path = os.getenv("LORA_PATH") or os.getenv("QWEN2AUDIO_LORA_PATH")
    model_cache_path = os.getenv("QWEN_CACHE_DIR")

    if not model_name:
        raise RuntimeError("MODEL_NAME not set in environment or .env")
    if not lora_path:
        raise RuntimeError("LORA_PATH not set in environment or .env")


    processor = AutoProcessor.from_pretrained(model_name, cache_dir=model_cache_path)

    model = Qwen2AudioForConditionalGeneration.from_pretrained(
        model_name, torch_dtype="auto", device_map="auto", cache_dir=model_cache_path
    )

    model = PeftModel.from_pretrained(model, lora_path)
    model.eval()

    return processor, model

In [13]:
processor, model = load_model()

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:02<00:00,  2.31it/s]


In [16]:
USER_PROMPT = """###Your task is to generate exact transcript of the given audio and return in a strict JSON strcuture
JSON SCHEMA:
{
"transcript": <exact trascription of input audio>
}
"""

In [17]:
convo = [
            {"role": "user", "content": [
                {"type": "text", "text": USER_PROMPT},
                {"type": "audio", "audio": "test.wav"},
            ]}
        ]

In [19]:
text = processor.apply_chat_template(convo, add_generation_prompt=True, tokenize=False)
inputs = processor(text=text, audio=wav, return_tensors="pt", padding=True, sampling_rate=16000)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

generate_ids = model.generate(**inputs, max_length=2048, temperature=0.001, num_beams=1, num_return_sequences=1)
generate_ids = generate_ids[:, inputs["input_ids"].size(1):]

In [21]:
response = processor.batch_decode(
    generate_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
    )

response[0]

"{'transcript': 'yeah, sure. okay. um i have around twelve thirteen fourteenths years experience on on it. fair's health i'll say seven years. i was working as a software engineer in in software development projects and after that i moved to data engineering.'}"

### TEST BARE

In [7]:
from transformers import Qwen2AudioForConditionalGeneration

MODEL_NAME = "Qwen/Qwen2-Audio-7B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_NAME, cache_dir="./hf_models")

model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    # dtype=torch.bfloat16,
    cache_dir="./hf_models",
    # attn_implementation="flash_attention_2",
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:02<00:00,  2.31it/s]


In [8]:
# Apply LoRA weights
model = PeftModel.from_pretrained(model, "/home/studio/work/models/checkpoint-9300")

In [9]:
USER_PROMPT = """
You are an expert Technical Interview Assistant. You are listening to an audio segment from a <SPEAKER> and provided with the previous conversation context.

TASK: Transcribe the audio exactly and evaluate the candidate's technical performance in real-time. Return JSON ONLY.

---
FIELD DEFINITIONS & LOGIC:

1. "transcript": 
   - Transcribe the audio EXACTLY as heard. 
   - Include all fillers (uhh, umm, mmh), stutters, and natural pauses. 
   - Do not autocorrect or normalize the speech.

2. "technical_qa": 
   - Set to true if the current audio segment contains technical content (coding, architecture, data, system design).
   - Set to false for intros, behavioral questions, or logistics.

3. "response_reasoning": 
   - (Candidate Only): Provide a cumulative evaluation of the candidate's answer so far. 
   - Explicitly mention if the current segment clarifies, completes, or contradicts previous segments of the same answer.
   - (Interviewer): null.

4. answer_rating:
   (Candidate only)
   Rate based on correctness AND completeness relative to the question.
   excellent:
   - fully correct
   - complete answer
   - includes key concepts and reasoning

   satisfactory:
   - partially correct OR missing depth
   - minor gaps but core idea is present

   poor:
   - incorrect answer
   - vague or generic response
   - misses key concept required by the question
   - does not answer the question directly
   - shows misunderstanding

   CRITICAL RULES:
   - If ANY key concept is missing → NOT excellent
   - If answer is vague, generic, or off-topic → poor
   - If answer does not clearly address the question → poor
   - If unsure between satisfactory and poor → choose poor
   - DO NOT default to satisfactory

5. "follow_up_question":
   - If technical_qa is true AND answer_rating is NOT "excellent", generate a clean, professional probe to dig deeper into the current topic.
   - If the candidate is mid-sentence, the question should anticipate what is missing.
   - Otherwise: null.

6. "update_follow_up_question":
   - Set to true ONLY when:
     a) The candidate starts a brand new technical topic.
     b) The candidate completes a thought or segment, requiring the previous follow-up question to be refreshed or removed.
     c) A previously generated follow-up question is no longer relevant based on the new audio.
   - Set to false for the interviewer and during seamless speech continuations.

---
STRICT JSON STRUCTURE:
{
  "transcript": "...",
  "technical_qa": boolean,
  "response_reasoning": "...",
  "answer_rating": "...",
  "follow_up_question": "...",
  "update_follow_up_question": boolean
}

PREVIOUS CONTEXT:
<PREVIOUS CONTEXT>
"""

In [10]:
conversation = [
            {"role": "user", "content": [
                {"type": "text", "text": USER_PROMPT.replace("<SPEAKER>", "Candidate").replace("<PREVIOUS CONTEXT>", "interviewer:And then, you know, I can ask you more questions?")},
                {"type": "audio", "audio": "test.wav"},
            ]}
        ]

In [11]:
text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
inputs = processor(text=text, audio=wav, return_tensors="pt", padding=True, sampling_rate=16000)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

generate_ids = model.generate(**inputs, max_length=2048, temperature=0.001, num_beams=1, num_return_sequences=1)
generate_ids = generate_ids[:, inputs["input_ids"].size(1):]

In [12]:
responses = processor.batch_decode(
        generate_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )
responses

['{"transcript":"yeah, sure. okay. um, i have around twelve thirteen, um thirteen fourteen years experience on on it. fair\'s health, i\'ll say seven years. i was working as a software engineer in in software development projects. um and after that i moved to data engineering.","is_technical_qa":false,"response_reasoning":"Candidate provides a rough timeline of their career history, focusing on software engineering and data engineering.","answer_rating":"satisfactory","follow_up_question":"Can you describe a data engineering project you worked on?","update_follow_up_question":true}']

In [1]:
def load_model() -> tuple:
    """Load Qwen2Audio processor and LoRA-adapted model.

    Reads `MODEL_NAME` and `LORA_PATH` from a `.env` file or environment.

    Returns:
        (processor, model)
    """
    # Load env vars if present
    try:
        load_dotenv()
    except Exception:
        # noop if dotenv not available
        pass

    model_name = os.getenv("MODEL_NAME") or os.getenv("QWEN2AUDIO_MODEL_NAME")
    lora_path = os.getenv("LORA_PATH") or os.getenv("QWEN2AUDIO_LORA_PATH")
    model_cache_path = os.getenv("QWEN_CACHE_DIR")

    if not model_name:
        raise RuntimeError("MODEL_NAME not set in environment or .env")
    if not lora_path:
        raise RuntimeError("LORA_PATH not set in environment or .env")

    LOGGER.info("Loading model and processor... model=%s lora=%s", model_name, lora_path)

    processor = AutoProcessor.from_pretrained(model_name, cache_dir=model_cache_path)

    model = Qwen2AudioForConditionalGeneration.from_pretrained(
        model_name, torch_dtype="auto", device_map="auto", cache_dir=model_cache_path
    )

    model = PeftModel.from_pretrained(model, lora_path)
    model.eval()

    LOGGER.info("Model and processor loaded successfully")
    return processor, model

In [ ]:
self.processor, self.model = load_model()